# Plot the white light curve for both visits

In [ ]:
from preamble_jax import *

In [ ]:
visit = 'S22'

orbits_to_exclude = np.array([0,2,7])
lc_index = 1 if visit=='F21' else 0
bin_edges = jnp.array([0.79,1.135,1.64]) * u.micron

predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']

rainbow = read_rainbow(f'{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy')
binned_for_model = rainbow.bin(wavelength_edges=bin_edges)
fleck_wavelengths = binned_for_model.wavelength.value

integrated_flux = np.trapz(rainbow.flux.value,x=rainbow.wavelength.value,axis=0)

img_date = rainbow.time.value
data_flux = integrated_flux/integrated_flux.mean()
relative_err = np.sqrt(integrated_flux/rainbow.nwave)/integrated_flux
time_from_T0 = img_date - predicted_T0

plt.figure(figsize=(9,3))
plt.errorbar(time_from_T0,data_flux,relative_err,
             label=f'Median Err = {int(np.nanmedian(relative_err)*1e6) }ppm',
            fmt='o',ms=1)
plt.legend()
plt.title(f'{visit} Integrated Light Curve')
plt.show()
plt.clf()

In [ ]:
# Label the orbits
orbit = np.zeros_like(img_date)
for j in range(len(img_date)):
    if j >= 1:
        if (img_date[j] - img_date[j - 1]) > 0.01:
            orbit[j] = (orbit[j - 1] + 1)
        else:
            orbit[j] = orbit[j - 1]

# Trim the first point from each orbit
ref_time = []
for o in np.unique(orbit):
    first_index = np.where(orbit == o)[0][0]
    ref_time.append(img_date[first_index])
    data_flux[first_index] = np.nan  # Set the first point of each subsequent orbit to np.nan
    relative_err[first_index] = np.nan
    img_date[first_index] = np.nan
    time_from_T0[first_index] = np.nan

# Set data to nan if it was in the pre-defined list of orbits to exclude
for orbit_to_exclude in orbits_to_exclude:
    data_flux[orbit == orbit_to_exclude] = np.nan
    relative_err[orbit == orbit_to_exclude] = np.nan
    img_date[orbit == orbit_to_exclude] = np.nan
    time_from_T0[orbit == orbit_to_exclude] = np.nan

plt.figure(figsize=(9,3) )
plt.errorbar(time_from_T0,data_flux,yerr=relative_err,fmt='o',ms=1)
plt.show()
plt.clf()

# Populate ramp_phase time arrays
phase_list=[]
for o in [0,1,2,3,4,5,6,7]:
    rphase = (img_date[orbit==o] - ref_time[o]) / 0.066
    phase_list.append(rphase)
ramp_phase = np.concatenate(phase_list)
breathing_phase = ( (img_date-ref_time[0]+0.02) / 0.066 ) % 1

In [ ]:
# Convert all arrays to JAX arrays
_data_flux = jnp.array( data_flux[~np.isnan(time_from_T0)] )
data_flux = _data_flux[~jnp.isnan(_data_flux)]
_relative_err = jnp.array(relative_err[~np.isnan(time_from_T0)] )
relative_err = _relative_err[~jnp.isnan(_data_flux)]
_img_date = jnp.array(img_date[~np.isnan(time_from_T0)] )
img_date = _img_date[~jnp.isnan(_data_flux)]
_time_from_T0 = jnp.array(time_from_T0[~np.isnan(time_from_T0)] )
time_from_T0 = _time_from_T0[~jnp.isnan(_data_flux)]
_ramp_phase = jnp.array(ramp_phase[~np.isnan(ramp_phase)] )
ramp_phase = _ramp_phase[~jnp.isnan(_data_flux)]
_breathing_phase = jnp.array(breathing_phase[~np.isnan(breathing_phase)] )
breathing_phase = _breathing_phase[~jnp.isnan(_data_flux)]

normalized_data_flux = data_flux/jnp.mean(data_flux)
normalized_relative_err = relative_err*normalized_data_flux

plt.figure(figsize=(9,3) )
plt.errorbar(time_from_T0,normalized_data_flux,yerr=normalized_relative_err,fmt='o',ms=1)
plt.show()
plt.clf()

In [ ]:
default_params={

    # Ramp Model Parameters
    'r_1': 18.1,
    'r_2': -6.7,
    'r_3': 0.0135,
    #Breathing params
    'b_1':0,
    'b_2':0,
    'b_3':0,
    'b_4':0,
    
    # Planet parameters
    'Mp':8.0,
    'i_planet':89.5,
    'P_orb':8.463,
    't_0':0.00048,
    'rp_rstar': 0.0465,
    'a_rstar': 19.15,
    'ecc': 0.0,
    'u_1':0.35,
    'u_2':0.15,
    
    # Stellar parameters
    'log_g':4.5,
    'i_star':89.5,
    'P_rot':4.86,
    'Rs':0.8,
    'Ms':0.6,
    'metallicity':0.0,

    # Spot Parameters
    # 'log_fixedspot_radii':-2.0,
    # 'T_unocculted': 3100,
    # 'T_occulted': 3500,
    'T_phot': 4100,
    'T_spot': 3500,
    'limb_spot_lon':-1.305,
    'limb_spot_lat':0.9,
    'limb_spot_rad':0.22,
    'chord_spot_lon_1':0.05,
    'chord_spot_lat_1':1.54,
    'chord_spot_rad_1':0.15,
    'chord_spot_lon_2':0.86,
    'chord_spot_lat_2':1.66,
    'chord_spot_rad_2':0.055,

    # Model meta-parameters
    'beta':0.25,

}

In [ ]:
def plot_results(parameter_dict,
                 samples=None,
                 figlabel='missinglabel'):
    params = parameter_dict

    # Calculate systematics model
    ramp = ramp_model_jax(phase=ramp_phase,
                    r1=jnp.asarray(params.get('r_1',default_params['r_1']), dtype=jnp.float64),
                    r2=jnp.asarray(params.get('r_2',default_params['r_2']), dtype=jnp.float64),
                    r3=jnp.asarray(params.get('r_3',default_params['r_3']), dtype=jnp.float64) )
    breathing = breathing_model_jax(phase=breathing_phase,
                              b1=jnp.asarray(params.get('b_1',default_params['b_1']), dtype=jnp.float64),
                              b2=jnp.asarray(params.get('b_2',default_params['b_2']), dtype=jnp.float64),
                              b3=jnp.asarray(params.get('b_3',default_params['b_3']), dtype=jnp.float64),
                              b4=jnp.asarray(params.get('b_4',default_params['b_4']), dtype=jnp.float64))
    _systematics = (breathing * ramp)
    systematics = _systematics/jnp.mean(_systematics)
    
    # Update Planet Parameters
    planet_parameters = dict(
        inclination = jnp.radians(default_params['i_planet']),
        a = jnp.asarray(params.get('a_rstar',default_params['a_rstar']), dtype=jnp.float64),
        rp = jnp.asarray(params.get('rp_rstar',default_params['rp_rstar']), dtype=jnp.float64),
        period = jnp.asarray(params.get('P_orb',default_params['P_orb']), dtype=jnp.float64),
        t0 = jnp.asarray(params.get('t_0',default_params['t_0']), dtype=jnp.float64),
        ecc = jnp.asarray(params.get('ecc',default_params['ecc']), dtype=jnp.float64),
        u1 = jnp.asarray(params.get('u_1',default_params['u_1']), dtype=jnp.float64),
        u2 = jnp.asarray(params.get('u_2',default_params['u_2']), dtype=jnp.float64),
    )
    
    # Get spectra for the model temps
    Phot = get_binned_BTSettl_spectrum_jax(T=jnp.asarray(params.get('T_phot',default_params['T_phot']), dtype=jnp.float64),
                                           data_wave = fleck_wavelengths)
    Cool = get_binned_BTSettl_spectrum_jax(T=jnp.asarray(params.get('T_spot',default_params['T_spot']), dtype=jnp.float64),
                                           data_wave = fleck_wavelengths)
    
    # Update stellar parameters
    active_star = ActiveStar(
        times = time_from_T0,
        inclination = jnp.radians(default_params['i_star']),
        T_eff=jnp.asarray(params.get('T_phot',default_params['T_phot']), dtype=jnp.float64),
        wavelength=Phot[0]*1e-6, # convert from micron to m
        phot=Phot[1],
        P_rot=default_params['P_rot']
    )
    
    active_star.spectrum = jnp.concatenate([
        jnp.vstack([Cool[1],Cool[1],Cool[1]
                   ]),
    ])
    active_star.temperature = jnp.concatenate([
        jnp.array([params.get('T_spot', default_params['T_spot']),
                   params.get('T_spot', default_params['T_spot']),
                   params.get('T_spot', default_params['T_spot'])
                  ]),
    ])
    active_star.lon = jnp.concatenate([
        jnp.array([params.get('limb_spot_lon', default_params['limb_spot_lon']), 
                   params.get('chord_spot_lon_1', default_params['chord_spot_lon_1']), 
                   params.get('chord_spot_lon_2', default_params['chord_spot_lon_2']),
                   ]),
    ])
    active_star.lat = jnp.concatenate([
        jnp.array([params.get('limb_spot_lat', default_params['limb_spot_lat']), 
                   params.get('chord_spot_lat_1', default_params['chord_spot_lat_1']), 
                   params.get('chord_spot_lat_2', default_params['chord_spot_lat_2']),
                   ]),
    ])
    active_star.rad = jnp.concatenate([
        jnp.array([params.get('limb_spot_rad', default_params['limb_spot_rad']), 
                   params.get('chord_spot_rad_1', default_params['chord_spot_rad_1']), 
                   params.get('chord_spot_rad_2', default_params['chord_spot_rad_2']),
                   ]),
    ])        
    
    lc, contam, X, Y, spectrum_at_transit = active_star.transit_model(**planet_parameters)        
    lc_model = lc.T[lc_index] * systematics
    mean_per_wavelength = jnp.mean(lc_model)
    normalized_model = lc_model / mean_per_wavelength
    residuals = normalized_data_flux - normalized_model
    adjusted_flux_err = (10**params.get('beta', default_params['beta']))*normalized_relative_err

    'Create a 3-panel figure'
    plt.figure(figsize=(12, 6))
    gs = plt.GridSpec(2, 2, width_ratios=[1, 2])
    ax_star = plt.subplot(gs[:, 0])  # All rows, first column
    active_star.plot_star(
        t0=params.get('t_0', default_params['t_0']),
        rp=params.get('rp_rstar', default_params['rp_rstar']),
        a=params.get('a_rstar', default_params['a_rstar']),
        inclination=np.radians(89.5),
        ecc=params.get('ecc', default_params['ecc']),
        ax=ax_star)
    # Top right panel - light curve
    ax_lc = plt.subplot(gs[0, 1])  # First row, second column
    ax_lc.errorbar(time_from_T0, normalized_data_flux,
                 yerr=adjusted_flux_err,
                 fmt='o', ms=1, color='k')
    ax_lc.scatter(time_from_T0, normalized_model, s=1, color='r')  
    ax_lc.set_ylabel('Relative Flux')
    ax_lc.set_title(f'{visit} Broadband Light Curve with Model')
    # Bottom right panel - residuals
    ax_res = plt.subplot(gs[1, 1], sharex=ax_lc)  # Second row, second column, shared x-axis
    ax_res.errorbar(time_from_T0, residuals*1e6,
                    yerr=adjusted_flux_err*1e6,
                    fmt='o', ms=1, color='k')
    ax_res.axhline(y=0, color='r', linestyle='-', alpha=0.5)
    ax_res.set_ylabel('Residual Flux (ppm)')
    ax_res.set_xlabel('Time from T0')
    ax_res.set_title('Data-Model')
    
    plt.tight_layout()
    plt.savefig(f'{model_designation}_{figlabel}.png', dpi=400)
    plt.show()
    plt.clf()

    return normalized_model,active_star

In [ ]:
# model_designation = 'S22_60000samples_12chains_3spots_whitelight_2025_12_23'
# model_designation = 'S22_60000samples_12chains_3spots_whitelight_2025_12_30' # current winner
model_designation = 'S22_100000samples_12chains_3spots_whitelight_final'

result = arviz.from_netcdf(f'{model_designation}')
'Print the summary'
arviz.summary(result)

'Make a corner plot'
corner.corner(
    result,
)
plt.savefig(f'{model_designation}_corner.png',dpi=200)

'Examine the Leave-One-Out (LOO) summary'
loo = arviz.loo(result, pointwise=True)
loo

In [ ]:
latex_summary = az.summary(result)
latex_summary.to_latex(f"{model_designation}-results-table.tex", float_format="%.6f")

results = []
for var_name in result.posterior.data_vars:
    samples = result.posterior[var_name].values.flatten()
    
    q16 = np.quantile(samples, 0.16)
    median = np.quantile(samples, 0.50)
    q84 = np.quantile(samples, 0.84)
    
    results.append({
        'parameter': var_name,
        'q16': q16,
        'median': median,
        'q84': q84,
        'lower_err': (median-q16),
        'upper_err':(q84-median)
    })

# Convert to DataFrame
quantile_df = pd.DataFrame(results)
quantile_df = quantile_df.set_index('parameter')

# Save to LaTeX
quantile_df.to_latex(f"{model_designation}-quantile_results.tex", float_format="%.6f")

## Print and plot the MEDIAN parameter results

In [ ]:
final_median_params = arviz.summary(result,stat_focus='median')['median']

## Print and plot the Maximum Likelihood Estimate results

In [ ]:
def calculate_model(parameter_dict,relative_flux):
    
    params = parameter_dict

    # Calculate systematics model
    ramp = ramp_model_jax(phase=ramp_phase,
                    r1=jnp.asarray(params.get('r_1',default_params['r_1']), dtype=jnp.float64),
                    r2=jnp.asarray(params.get('r_2',default_params['r_2']), dtype=jnp.float64),
                    r3=jnp.asarray(params.get('r_3',default_params['r_3']), dtype=jnp.float64) )
    breathing = breathing_model_jax(phase=breathing_phase,
                              b1=jnp.asarray(params.get('b_1',default_params['b_1']), dtype=jnp.float64),
                              b2=jnp.asarray(params.get('b_2',default_params['b_2']), dtype=jnp.float64),
                              b3=jnp.asarray(params.get('b_3',default_params['b_3']), dtype=jnp.float64),
                              b4=jnp.asarray(params.get('b_4',default_params['b_4']), dtype=jnp.float64))

    _systematics = (breathing * ramp)
    systematics = _systematics/jnp.mean(_systematics)
    
    # Update Planet Parameters
    planet_parameters = dict(
        inclination = jnp.radians(default_params['i_planet']),
        a = jnp.asarray(params.get('a_rstar',default_params['a_rstar']), dtype=jnp.float64),
        rp = jnp.asarray(params.get('rp_rstar',default_params['rp_rstar']), dtype=jnp.float64),
        period = jnp.asarray(params.get('P_orb',default_params['P_orb']), dtype=jnp.float64),
        t0 = jnp.asarray(params.get('t_0',default_params['t_0']), dtype=jnp.float64),
        ecc = jnp.asarray(params.get('ecc',default_params['ecc']), dtype=jnp.float64),
        u1 = jnp.asarray(params.get('u_1',default_params['u_1']), dtype=jnp.float64),
        u2 = jnp.asarray(params.get('u_2',default_params['u_2']), dtype=jnp.float64),
    )
    no_planet_params = dict(
        inclination = jnp.radians(default_params['i_planet']),
        a = jnp.asarray(params.get('a_rstar',default_params['a_rstar']), dtype=jnp.float64),
        rp = 0.0,
        period = jnp.asarray(params.get('P_orb',default_params['P_orb']), dtype=jnp.float64),
        t0 = jnp.asarray(params.get('t_0',default_params['t_0']), dtype=jnp.float64),
        ecc = jnp.asarray(params.get('ecc',default_params['ecc']), dtype=jnp.float64),
        u1 = jnp.asarray(params.get('u_1',default_params['u_1']), dtype=jnp.float64),
        u2 = jnp.asarray(params.get('u_2',default_params['u_2']), dtype=jnp.float64),
    )
    
    # Get spectra for the model temps
    Phot = get_binned_BTSettl_spectrum_jax(T=jnp.asarray(params.get('T_phot',default_params['T_phot']), dtype=jnp.float64),
                                           data_wave = fleck_wavelengths)
    Cool = get_binned_BTSettl_spectrum_jax(T=jnp.asarray(params.get('T_spot',default_params['T_spot']), dtype=jnp.float64),
                                           data_wave = fleck_wavelengths)
    
    # Update stellar parameters
    active_star = ActiveStar(
        times = time_from_T0,
        inclination = jnp.radians(default_params['i_star']),
        T_eff=jnp.asarray(params.get('T_phot',default_params['T_phot']), dtype=jnp.float64),
        wavelength=Phot[0]*1e-6, # convert from micron to m
        phot=Phot[1],
        P_rot=default_params['P_rot']
    ) 
    
    active_star.spectrum = jnp.concatenate([
        jnp.vstack([Cool[1],Cool[1],Cool[1]
                   ]),
    ])
    active_star.temperature = jnp.concatenate([
        jnp.array([params.get('T_spot', default_params['T_spot']),
                   params.get('T_spot', default_params['T_spot']),
                   params.get('T_spot', default_params['T_spot'])
                  ]),
    ])
    active_star.lon = jnp.concatenate([
        jnp.array([params.get('limb_spot_lon', default_params['limb_spot_lon'])+0.16, 
                   params.get('chord_spot_lon_1', default_params['chord_spot_lon_1']), 
                   params.get('chord_spot_lon_2', default_params['chord_spot_lon_2']),
                   ]),
    ])
    active_star.lat = jnp.concatenate([
        jnp.array([1.2,#params.get('limb_spot_lat', default_params['limb_spot_lat']), 
                   params.get('chord_spot_lat_1', default_params['chord_spot_lat_1']), 
                   params.get('chord_spot_lat_2', default_params['chord_spot_lat_2']),
                   ]),
    ])
    active_star.rad = jnp.concatenate([
        jnp.array([0.87*params.get('limb_spot_rad', default_params['limb_spot_rad']), 
                   params.get('chord_spot_rad_1', default_params['chord_spot_rad_1']), 
                   params.get('chord_spot_rad_2', default_params['chord_spot_rad_2']),
                   ]),
    ])

    fig,star_ax = plt.subplots(1,1,figsize=(6,6))
    active_star.plot_star(
        t0=params.get('t_0', default_params['t_0']),
        rp=params.get('rp_rstar', default_params['rp_rstar']),
        a=params.get('a_rstar', default_params['a_rstar']),
        inclination=np.radians(89.5),
        ecc=params.get('ecc', default_params['ecc']),ax=star_ax)
    plt.savefig('S22_starmap.png',dpi=600) 
    plt.show()
    plt.clf()
    
    lc, contam, X, Y, spectrum_at_transit = active_star.transit_model(**planet_parameters)   
    rotation_ = active_star.transit_model(**no_planet_params)[0]

    lc_model = lc.T[lc_index] * systematics
    mean_per_wavelength = jnp.mean(lc_model)

    rot_model = rotation_.T[lc_index]
    mean_rot = jnp.mean(rot_model)
    rotation_curve = rot_model/mean_rot

    normalized_model = lc_model / mean_per_wavelength
    de_trended_model = normalized_model / systematics
    de_trended_data = relative_flux/systematics

    de_trended_model /= jnp.mean(de_trended_model)
    de_trended_data /= jnp.mean(de_trended_data)

    residuals = de_trended_data - de_trended_model

    return normalized_model, residuals, de_trended_model, de_trended_data, rotation_curve

def get_representative_sample_from_dict(posterior_dict_entry):

    posterior_dataset = posterior_dict_entry
    
    medians = posterior_dataset.median()
    
    param_names = list(posterior_dataset.data_vars)

    all_samples = []
    median_values = []
    
    for param in param_names:
        samples = posterior_dataset[param].values.flatten()
        all_samples.append(samples)
        median_values.append(float(medians[param]))
    
    param_matrix = np.column_stack(all_samples)  # n_samples x n_params
    median_array = np.array(median_values)       # n_params
    
    # Find sample closest to medians
    distances = np.linalg.norm(param_matrix - median_array, axis=1)
    closest_idx = np.argmin(distances)
    
    representative_params = {}
    for i, param in enumerate(param_names):
        representative_params[param] = param_matrix[closest_idx, i]
    
    return representative_params

def plot_final_lightcurve(jax_flux, jax_rel_err, time_from_T0, 
                           posterior_samples_dict, designation):
    
    fig, (ax_lc, ax_sys_removed, ax_transit, ax_res) = plt.subplots(4, 1, figsize=(8, 10), sharex=True)
    
    param_medians = get_representative_sample_from_dict(posterior_samples_dict[0])
    print(param_medians)
    normed_model, residuals, median_model, detrended_relative_flux, rotation_curve = calculate_model(param_medians, jax_flux)


    'First row - data and model'
    ax_lc.plot(time_from_T0[0:19], normed_model[0:19], '-', 
                color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_lc.plot(time_from_T0[19:38], normed_model[19:38], '-', 
                color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_lc.plot(time_from_T0[38:57], normed_model[38:57], '-', 
                color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_lc.plot(time_from_T0[57:76], normed_model[57:76], '-', 
                color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_lc.plot(time_from_T0[76:95], normed_model[76:95], '-', 
                color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_lc.plot(time_from_T0[95:], normed_model[95:], '-', 
                color='k', linewidth=2, alpha=1.0, zorder=-100,label='Model')
    ax_lc.errorbar(time_from_T0, jax_flux,
                    yerr=jax_rel_err*jax_flux, fmt='o', alpha=0.3,
                    # color='blue',
                    markeredgecolor='k', markeredgewidth=0.4,label='Data')
    ax_lc.set_title('S22/G102 Astrophysical White Light Model',fontsize=16)
    ax_lc.set_ylabel('Relative Flux', fontsize=12)
    ax_lc.legend(loc='upper right')
    
    'Second row - data and model with systematics removed'
    ax_sys_removed.errorbar(time_from_T0, detrended_relative_flux,
                    yerr=jax_rel_err*detrended_relative_flux, fmt='o', alpha=0.3,
                    # color='blue',
                    markeredgecolor='k', markeredgewidth=0.4)
    ax_sys_removed.plot(time_from_T0[0:19], median_model[0:19], '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_sys_removed.plot(time_from_T0[19:38], median_model[19:38], '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_sys_removed.plot(time_from_T0[38:57], median_model[38:57], '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_sys_removed.plot(time_from_T0[57:76], median_model[57:76], '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_sys_removed.plot(time_from_T0[76:95], median_model[76:95], '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_sys_removed.plot(time_from_T0[95:], median_model[95:], '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_sys_removed.set_title('Instrument-Detrended',fontsize=14)
    ax_sys_removed.set_ylabel('Relative Flux', fontsize=12)

    'Third row - data and model with systematics removed'
    ax_transit.errorbar(time_from_T0, detrended_relative_flux/rotation_curve-0.001,
                    yerr=jax_rel_err*(detrended_relative_flux/rotation_curve), fmt='o', alpha=0.3,
                    # color='blue',
                    markeredgecolor='k', markeredgewidth=0.4)
    ax_transit.plot(time_from_T0[0:19], (median_model/rotation_curve)[0:19]-0.001, '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_transit.plot(time_from_T0[19:38], (median_model/rotation_curve)[19:38]-0.001, '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_transit.plot(time_from_T0[38:57], (median_model/rotation_curve)[38:57]-0.001, '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_transit.plot(time_from_T0[57:76], (median_model/rotation_curve)[57:76]-0.001, '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_transit.plot(time_from_T0[76:95], (median_model/rotation_curve)[76:95]-0.001, '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_transit.plot(time_from_T0[95:], (median_model/rotation_curve)[95:]-0.001, '-', 
                    color='k', linewidth=2, alpha=1.0, zorder=-100)
    ax_transit.set_title('Rotation-Detrended',fontsize=14)
    ax_transit.set_ylabel('Relative Flux', fontsize=12)

    'Fourth row - Residuals'
    ax_res.errorbar(time_from_T0, residuals, 
                    yerr=jax_rel_err, fmt='o', 
                    # color='blue',
                    alpha=0.4)
    ax_res.axhline(0,color='k')
    ax_res.set_title('Residual Flux',fontsize=14)
    ax_res.set_ylabel('Data-Model', fontsize=12)
    ax_res.set_xlabel('Time from T0 (d)', fontsize=12)

    fig.tight_layout()
    
    # Save figure
    plt.savefig(f'{model_designation}_detrended_final_model.png', dpi=400)
    plt.show()
    
posterior_samples = {}
result = arviz.InferenceData.from_netcdf(f'{model_designation}')
posterior_samples[0] = result.posterior

scaling_factor = jnp.array([
    10**float(posterior_samples[0]['beta'].median().values)
])

jax_rel_err = scaling_factor * normalized_relative_err
jax_flux = normalized_data_flux

plot_final_lightcurve(jax_flux, jax_rel_err, time_from_T0,
    posterior_samples, model_designation)